# 04 - ETL Run Summary

Review generated raw files, processed warehouse files, data quality results, output files, and optional PostgreSQL row counts.

## Setup Project Paths

Prepare folder references for raw, processed, and output CSV summaries.

In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'output'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

## Summarize CSV Outputs

Count rows and columns across `data/raw/`, `data/processed/`, and `output/`.

**Output:** file-level row and column summary.

In [2]:
def summarize_csv(folder):
    rows = []
    for path in sorted(folder.glob('*.csv')):
        df = pd.read_csv(path)
        rows.append({'folder': folder.name, 'file': path.name, 'rows': len(df), 'columns': len(df.columns)})
    return pd.DataFrame(rows)

etl_summary_processed_tables = pd.concat([summarize_csv(RAW_DIR), summarize_csv(PROCESSED_DIR), summarize_csv(OUTPUT_DIR), summarize_csv(VALIDATION_DIR)], ignore_index=True)
etl_summary_processed_tables.to_csv(VALIDATION_DIR / 'etl_summary_processed_tables.csv', index=False)
etl_summary_processed_tables

,folder,file,rows,columns
0,raw,customers.csv,1000,11
1,raw,inventory.csv,2500,7
2,raw,payments.csv,2000,5
3,raw,products.csv,100,9
4,raw,promotions.csv,21,6
5,raw,sales_details.csv,4688,7
6,raw,sales_transactions.csv,2000,9
7,raw,stores.csv,25,7
8,processed,dim_channel.csv,3,4
9,processed,dim_customer.csv,1000,10


## Review Data Quality Results

Display `output/dq_summary.csv` from Notebook 02. If the file is missing, run `02_transform_validate.ipynb` first.

In [3]:
dq_path = OUTPUT_DIR / 'dq_summary.csv'
if dq_path.exists():
    display(pd.read_csv(dq_path))
else:
    print('dq_summary.csv is not available yet. Run 02_transform_validate.ipynb first.')

,check_name,failed_records,severity,status,checked_at
0,Duplicate transaction_id,0,critical,PASS,2026-05-31T01:30:56
1,Duplicate detail_id,0,critical,PASS,2026-05-31T01:30:56
2,Quantity must be > 0,0,critical,PASS,2026-05-31T01:30:56
3,Unit price must be non-negative,0,critical,PASS,2026-05-31T01:30:56
4,Gross sales equals quantity * unit_price,0,critical,PASS,2026-05-31T01:30:56
5,Net sales must be non-negative,0,critical,PASS,2026-05-31T01:30:56
6,Product maps to dim_product,0,critical,PASS,2026-05-31T01:30:56
7,Missing customer handled as Guest Customer,0,major,PASS,2026-05-31T01:30:56
8,Stock quantity must be non-negative,0,critical,PASS,2026-05-31T01:30:56
9,Source net sales equals fact_sales net sales,0,critical,PASS,2026-05-31T01:30:56


## Summarize PostgreSQL Table Counts

Optionally connect to PostgreSQL and count key OLTP, warehouse, and OLAP tables. If the database is unavailable, only this database summary is skipped.

In [4]:
def get_database_url():
    load_dotenv(PROJECT_ROOT / '.env')
    if os.getenv('DATABASE_URL'):
        return os.getenv('DATABASE_URL')
    return f"postgresql+psycopg2://{os.getenv('DB_USER','postgres')}:{os.getenv('DB_PASSWORD','postgres')}@{os.getenv('DB_HOST','localhost')}:{os.getenv('DB_PORT','5432')}/{os.getenv('DB_NAME','erajaya_dw')}"

tables = ['oltp.tb_customer','oltp.tb_product','oltp.tb_store','oltp.tb_sales_transaction','oltp.tb_sales_detail','dw.dim_customer','dw.dim_product','dw.dim_store','dw.fact_sales','dw.fact_inventory_snapshot','olap.agg_monthly_sales','olap.mart_sales_overview']
try:
    engine = create_engine(get_database_url(), pool_pre_ping=True)
    with engine.connect() as conn:
        db_counts = [{'table': table, 'rows': conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar_one()} for table in tables]
    display(pd.DataFrame(db_counts))
except Exception as exc:
    print('Database summary skipped:', exc)

,table,rows
0,oltp.tb_customer,1000
1,oltp.tb_product,100
2,oltp.tb_store,25
3,oltp.tb_sales_transaction,2000
4,oltp.tb_sales_detail,4688
5,dw.dim_customer,1000
6,dw.dim_product,100
7,dw.dim_store,25
8,dw.fact_sales,4688
9,dw.fact_inventory_snapshot,2500


## Overall ETL Status

Buat ringkasan status akhir berdasarkan file validation. Status dianggap `PASS` jika tidak ada failed records pada data quality dan tidak ada invalid FK/numeric/table validation.

In [5]:
status_rows = []
validation_checks = {
    'extract_profile_summary.csv': ('is_extract_valid', 'Extract profile validity'),
    'transform_table_validation.csv': ('is_table_valid', 'Transform table validity'),
    'transform_numeric_validation.csv': ('is_numeric_valid', 'Transform numeric validity'),
    'transform_fk_validation.csv': ('is_fk_valid', 'Transform FK validity'),
}
for filename, (status_column, description) in validation_checks.items():
    path = VALIDATION_DIR / filename
    if path.exists():
        df = pd.read_csv(path)
        passed = bool(df[status_column].all()) if status_column in df.columns and len(df) else False
        failed_count = int((~df[status_column].astype(bool)).sum()) if status_column in df.columns and len(df) else 1
        status_rows.append({'validation_file': filename, 'description': description, 'failed_check_count': failed_count, 'status': 'PASS' if passed else 'FAIL'})
    else:
        status_rows.append({'validation_file': filename, 'description': description, 'failed_check_count': 1, 'status': 'MISSING'})

dq_path = VALIDATION_DIR / 'dq_summary.csv'
if dq_path.exists():
    dq = pd.read_csv(dq_path)
    failed_count = int((dq['status'] != 'PASS').sum())
    status_rows.append({'validation_file': 'dq_summary.csv', 'description': 'Data quality checks', 'failed_check_count': failed_count, 'status': 'PASS' if failed_count == 0 else 'FAIL'})
else:
    status_rows.append({'validation_file': 'dq_summary.csv', 'description': 'Data quality checks', 'failed_check_count': 1, 'status': 'MISSING'})

overall = pd.DataFrame(status_rows)
overall_status = 'PASS' if (overall['status'] == 'PASS').all() else 'CHECK_REQUIRED'
overall.loc[len(overall)] = {'validation_file': 'OVERALL', 'description': 'Overall ETL validation status', 'failed_check_count': int((overall['status'] != 'PASS').sum()), 'status': overall_status}
overall.to_csv(VALIDATION_DIR / 'etl_summary_overall_status.csv', index=False)
overall

,validation_file,description,failed_check_count,status
0,extract_profile_summary.csv,Extract profile validity,0,PASS
1,transform_table_validation.csv,Transform table validity,0,PASS
2,transform_numeric_validation.csv,Transform numeric validity,0,PASS
3,transform_fk_validation.csv,Transform FK validity,0,PASS
4,dq_summary.csv,Data quality checks,0,PASS
5,OVERALL,Overall ETL validation status,0,PASS
